# Alternating least squares for Tucker model

```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

The function `pyttb.tucker_als()` computes the best rank-$(R_1,R_2,\ldots,R_n)$ approximation of tensor $\mathcal{X}$, according to the specified dimensions in the vector $(R_1,R_2,\ldots,R_n)$. The form of $\mathcal{X}$ can be a `tensor`, `sptensor`, `ktensor`, or `ttensor`. The result is a `ttensor`.

The method is originally from Tucker (1966) and later revisited in De Lathauwer et al. (2000).

* L. R. Tucker, Some mathematical notes on three-mode factor analysis, Psychometrika, 31:279-311, 1966, http://dx.doi.org/10.1007/BF02289464
* L. De Lathauwer, B. De Moor, J. Vandewalle, On the best rank-1 and rank-(R_1, R_2, R_N) approximation of higher-order tensors, SIAM J. Matrix Analysis and Applications, 21:1324-1342, 2000, http://doi.org/10.1137/S0895479898346995

Note: Oftentimes it's better to use `pyttb.hosvd()` instead.

In [ ]:
from __future__ import annotations

import os
import sys

import numpy as np

import pyttb as ttb

## Problem setup

### Generate a noise-free data tensor
To make the results easy to interpret, we use the same data as in the [HOSVD tutorial](algorithm_hosvd.ipynb). We create a core `tensor` that is nearly block diagonal, with blocks whose norms decrease exponentially (using the function `tenrandblk()`), and embed it into a larger tensor `X` using orthogonal transformations. The multilinear rank of `X` is equal to the shape of the core.

In [ ]:
def tenrandblk(verbose=True):
    np.random.seed(0)
    # Block shapes (need not be cubic). Number of rows is the number
    # of levels and number of columns is the order of the tensor.
    bsz = np.array([[3, 2, 1], [2, 2, 2], [2, 3, 4]])

    # Squared norm of each block. Must be length L and sum to <= 1
    bns = np.array([0.9, 0.09, 0.009])

    # Find the end of each block
    bend = np.cumsum(bsz, 0)
    # Extract shape: D = # dimensions, L = # levels
    D, L = np.shape(bsz)
    # Final shape
    gsz = bend[-1, :]

    ## Create tensor
    # Figure out norm of off-block-diagonal
    dltnrmsqr = 1 - np.sum(bns)
    # Create pattern for off-block-diagonal to be modified as we go
    dltpattern = np.ones(tuple(gsz))
    # Create tensor to fill in
    G = ttb.tenzeros(tuple(gsz))

    # Create random entries to use
    A = np.sign(np.random.randn(*gsz))
    B = 0.1 * np.random.rand(*gsz) + 0.9
    Grnd = ttb.tensor(
        np.sign(np.random.randn(*gsz)) * (0.1 * np.random.rand(*gsz) + 0.9)
    )

    # Loop through and create blocks
    for i in range(L):
        # Figure out ith block pattern
        blkrange = []
        for k in range(D):
            if i == 0:
                blkrange.append(np.arange(bend[i, k]))
            else:
                blkrange.append(np.arange(bend[i - 1, k], bend[i, k]))

        # Create pattern that has ones for the block
        pattern = np.zeros(tuple(gsz))
        ix = np.ix_(blkrange[2], blkrange[0], blkrange[1])
        pattern[ix] = 1

        # Zero out block in the off-diagonal pattern
        dltpattern[ix] = 0

        # Randomly fill delta-pattern and rescale
        block = Grnd * pattern
        sse = (block**2).collapse()
        block *= np.sqrt(bns[i] / sse)

        # Add to main tensor
        G += block

        # Verbose output
        if verbose:
            print(
                f"Created block with shape {tuple(bsz[i, :])} with norm ({block.norm()})^2 = {block.norm() ** 2}"
            )

    if dltnrmsqr > 0:
        # final pattern
        block = Grnd * dltpattern
        sse = (block**2).collapse()
        block *= np.sqrt(dltnrmsqr / sse)
        G += block
        if verbose:
            print(
                f"Created tensor with shape {tuple(gsz)} with off-block-diagonal norm ({block.norm()})^2 = {block.norm() ** 2}"
            )

    return G, bsz, bns

In [ ]:
# Create core tensor with given block structure and norm 1
G, bsz, bns = tenrandblk()

In [ ]:
from scipy.stats import ortho_group


def mat_rand_orth(N, seed=0):
    # Generates random n x n orthogonal real matrix.
    return ortho_group.rvs(N, seed)


# shape of X
xsz = np.array([20, 20, 20])

# Create orthogonal matrices
U = []
for k in np.arange(3):
    V = mat_rand_orth(xsz[k])[:, : G.shape[k]]
    U.append(V)

# Create X
X = ttb.ttensor(G, U).full()

# The norm should be unchanged
print(f"||X|| = {X.norm()}")

### Generate noisy data
Real data is rarely exactly low multilinear rank. To mimic this, we add independent Gaussian noise to `X`. Because $\|\mathcal{X}\| = 1$, choosing the standard deviation of the noise to be $\sigma = \eta / \sqrt{N}$, where $N$ is the number of entries of `X`, gives noise with norm of about $\eta$, i.e., a relative noise level of $\eta$. We use $\eta = 0.05$ (5% noise).

In [ ]:
eta = 0.05  # relative noise level
np.random.seed(5)
X_noisy = ttb.tensor(np.random.normal(loc=X.data, scale=eta / np.sqrt(X.data.size)))
print(f"Relative noise: {(X_noisy - X).norm() / X.norm():.3f}")

## Basic usage

### Create an approximation with all ranks equal to 2

In [ ]:
T = ttb.tucker_als(X, 2)  # best rank(2,2,2) approximation
T

## Main options

### Create an approximation with specific ranks
The first block of the core has shape $(3, 2, 1)$ and contains 90% of the squared norm of `X`, so an approximation with ranks `[3, 2, 1]` already captures much of the tensor.

In [ ]:
T = ttb.tucker_als(X, [3, 2, 1])  # best rank(3,2,1) approximation
T

### Use a different ordering of the dimensions

In [ ]:
T = ttb.tucker_als(X, 2, dimorder=[2, 1, 0])
T

### Use the `"nvecs"` initialization method
This initialization is more expensive but generally works very well.

In [ ]:
T = ttb.tucker_als(X, 2, dimorder=[0, 1, 2], init="nvecs")
T

### Specify the initial guess manually

In [ ]:
U0 = [np.random.rand(20, 2), np.random.rand(20, 2), np.random.rand(20, 2)]
T = ttb.tucker_als(X, 2, dimorder=[0, 1, 2], init=U0)
T

### Use a sparse tensor as input
The input can also be a `sptensor`, `ktensor`, or `ttensor`. Here we convert a (dense) `tensor` to an `sptensor`.

In [ ]:
T_sparse, _, _ = ttb.tucker_als(X.to_sptensor(), [3, 2, 1], printitn=0)
T_sparse.core.shape

### Compute a decomposition of the noisy data
We compute decompositions of the noise-free and the noisy data with ranks equal to the block structure of the core. For the true multilinear rank `[7, 7, 7]`, the noise-free data are fit exactly. A Tucker model with ranks $(r_1, \ldots, r_d)$ has about $p = \prod_k r_k + \sum_k (I_k r_k - r_k^2)$ free parameters, which can absorb a small part of the noise, so we expect `normresidual` for the noisy data to be approximately $\|\mathcal{N}\|\sqrt{(N - p) / N}$, where $\|\mathcal{N}\|$ is the norm of the noise and $N$ is the number of entries of the tensor.

In [ ]:
N = X.data.size  # number of entries
noise_norm = (X_noisy - X).norm()
print(
    f"{'ranks':>10} {'fit (noise-free)':>17} {'fit (noisy)':>12} {'normresidual':>13} {'expected':>9}"
)
for ranks in [[3, 2, 1], [5, 4, 3], [7, 7, 7]]:
    _, _, output_clean = ttb.tucker_als(X, ranks, printitn=0)
    _, _, output_noisy = ttb.tucker_als(X_noisy, ranks, printitn=0)
    p = np.prod(ranks) + sum(s * r - r**2 for s, r in zip(X.shape, ranks))
    expected = noise_norm * np.sqrt((N - p) / N)
    if ranks == [7, 7, 7]:  # the noise-free data has no other source of residual
        expected_str = f"{expected:9.3f}"
    else:
        expected_str = f"{'':>9}"
    print(
        f"{str(ranks):>10} {output_clean['fit']:17.3f} {output_noisy['fit']:12.3f} "
        f"{output_noisy['normresidual']:13.3f} {expected_str}"
    )

## Interpreting the output

`tucker_als` returns the Tucker model `M` as a `ttensor`, the initial guess `Minit`, and a dictionary `output` with the number of iterations (`iters`), the norm of the residual (`normresidual`), the fit (`fit`, the fraction of the data explained by the model), and the parameters used (`params`).

In [ ]:
M, Minit, output = ttb.tucker_als(X, [3, 2, 1], printitn=0)
print(f"Core shape: {M.core.shape}")
print(f"Factor matrix shapes: {[f.shape for f in M.factor_matrices]}")
print(
    f"Iterations: {output['iters']}, "
    f"residual norm: {output['normresidual']:.3f}, "
    f"fit: {output['fit']:.3f}"
)

## Worked example

Create a `ttensor` with a `2 x 2 x 2` core and orthonormal factor matrices, and generate a noisy dense data tensor from it by sampling each entry from a normal distribution with standard deviation one and mean equal to the corresponding entry of the `ttensor`. Then compute a Tucker decomposition of the noisy data with the true multilinear rank, and compare the result to the known `ttensor`.

In [ ]:
np.random.seed(1)
shape = (8, 6, 4)
factors = [np.linalg.qr(np.random.rand(n, 2))[0] for n in shape]
core = ttb.tensor(30 * np.random.rand(2, 2, 2))
T_true = ttb.ttensor(core, factors)
X_noisy = ttb.tensor(np.random.normal(loc=T_true.full().data, scale=1.0))

T_fit, Minit, output = ttb.tucker_als(X_noisy, [2, 2, 2], init="nvecs", printitn=0)
print(f"Fit: {output['fit']:.3f}")
rel_err = (T_fit.full() - T_true.full()).norm() / T_true.full().norm()
print(f"Relative error against the true solution: {rel_err:.3f}")